# HIFA — Train trên Google Colab (GPU)

Notebook chạy các kiến trúc có LLM (A1 FLAG-Trader, B1 HARLF, C1 Debate, HIFA) trên GPU Colab.
Trước khi chạy: **Runtime → Change runtime type → GPU (T4/A100)**.

## ⚠ﺏ Chiến lược chống mất tiến độ (Colab Pro)
Train 5 seed mất ~22 giờ — **có thể bị Colab ngắt giữa chừng**. Notebook này đã cấu hình để:
1. **Mount Google Drive SỚM** và trỏ `checkpoints/` + `logs/` **thẳng vào Drive** (symlink).
2. Script train tự **lưu checkpoint mỗi 10 update** (`save_every`) + **tự resume** từ update đang dở.
3. Nếu Colab ngắt: chỉ cần **chạy lại toàn bộ notebook** từ đầu — training sẽ tự tiếp tục đúng chỗ đã dừng, không mất quá 10 update.

Xem hướng dẫn chi tiết ở `COLAB_GUIDE.md`.

## 1. Kiểm tra GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'KHÔNG thấy GPU! Vào Runtime → Change runtime type → GPU rồi chạy lại.'
print('CUDA available:', torch.cuda.is_available())
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Mount Google Drive SỚM (QUAN TRỌNG — phải chạy trước khi train)
Mount Drive ngay đầu phiên. Tất cả checkpoint + log sẽ được ghi trực tiếp vào thư mục này trên Drive,
nên **không bao giờ mất khi Colab ngắt phiên**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Thư mục bền vững trên Drive — sống sót qua mọi lần Colab reset.
DRIVE_ROOT = '/content/drive/MyDrive/LUAN_VAN/hifa_results'
import os
for sub in ('checkpoints', 'logs', 'reports'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
print('Drive sẵn sàng:', DRIVE_ROOT)

## 3. Lấy code
Sửa URL repo của bạn (Cách A - GitHub). Nếu dùng Drive, xem COLAB_GUIDE.md mục 1 Ô 2.

In [ ]:
%cd /content
![ -d hifa ] || git clone https://github.com/<username>/hifa.git
%cd /content/hifa
# Set PYTHONPATH một lần cho MỌI cell `!python` phía dưới.
%env PYTHONPATH=/content/hifa

## 4. Trỏ checkpoints/ + logs/ THẲNG vào Drive (symlink)
Sau cell này, mọi thứ script ghi vào `checkpoints/` và `logs/` thực chất nằm trên Drive.
Checkpoint resume (`checkpoints/phase_a/seed_N/resume.pt`) được lưu mỗi 10 update → an toàn tuyệt đối.

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/LUAN_VAN/hifa_results'
for name in ('checkpoints', 'logs', 'reports'):
    local = f'/content/hifa/{name}'
    target = f'{DRIVE_ROOT}/{name}'
    # Xóa thư mục local trống (nếu có) rồi tạo symlink trỏ sang Drive.
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        import shutil
        # Nếu local đã có dữ liệu (clone mới), copy sang Drive rồi thay bằng symlink.
        for f in os.listdir(local):
            src, dst = f'{local}/{f}', f'{target}/{f}'
            if not os.path.exists(dst):
                (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dst)
        shutil.rmtree(local)
    os.symlink(target, local)
    print(f'{local} -> {target}')

## 5. Cài dependencies (KHÔNG cài lại torch)
Chỉ báo kỹ thuật đã tự implement bằng pandas thuần (`src/data/build_indicators.py`),
nên KHÔNG cần `pandas-ta` nữa → dùng pandas/numpy bản mới nhất của Colab, không phải ghim version.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes \
    gymnasium stable-baselines3 yfinance quantstats \
    pyyaml scipy tensorboard pyarrow
# Colab cài sẵn torchao==0.10.0 (quá cũ) -> PEFT raise ImportError khi inject LoRA.
# Ta KHÔNG dùng quantization qua torchao nên gỡ bỏ cho sạch.
!pip uninstall -y -q torchao
import pandas, numpy
print('pandas:', pandas.__version__, '| numpy:', numpy.__version__)

## 6. Lấy dữ liệu Mỹ (fetch qua yfinance, ~2 phút)

In [ ]:
!python -m src.data.fetch_ohlcv_us --phase 0 --benchmark both
!python -m src.data.build_dataset --phase 0

## 7. Smoke test LLM thật trên GPU (verify trước khi train dài)

In [ ]:
!python scripts/_smoke_test_llm_real.py

## 8. Train A1 (FLAG-Trader-style), 5 seed — TỰ RESUME
Mỗi seed tự lưu checkpoint vào Drive mỗi 10 update. Nếu phiên bị ngắt, **chạy lại toàn bộ notebook**:
- Seed đã train xong → bỏ qua ngay (đã có `resume.pt` đủ `total_updates`).
- Seed đang dở → nạp lại weights + optimizer và chạy tiếp từ update dở.

`--seed {seed}` chạy từng seed riêng biệt (tiến trình độc lập, dễ theo dõi). Flag `--no-resume` nếu muốn train lại từ đầu.

In [ ]:
for seed in range(5):
    print(f'=== Training A1 seed {seed} (tự resume nếu có checkpoint) ===')
    !python -m src.training.train_phase_a --config configs/phase_a.yaml --seed {seed}

## 9. Baseline tài chính (B1 Index / B2 / B3 / B4 PPO) trên data Mỹ

In [ ]:
!python scripts/_run_baselines_us_phase0.py

## 10. Theo dõi training bằng TensorBoard (tùy chọn)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 11. (Tùy chọn) Backup thêm reports về Drive
`checkpoints/` và `logs/` đã symlink thẳng vào Drive nên tự động được lưu. Cell này chỉ dùng khi muốn
copy thêm bản rời (vd. reports sinh ngoài symlink). Thường KHÔNG cần nữa.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/LUAN_VAN/hifa_results'
!cp -r reports/* "{DRIVE_ROOT}/reports/" 2>/dev/null || true
print('Checkpoints + logs đã tự động nằm trên Drive (symlink). Reports đã backup.')